# Regime Overview

Visual sanity-check that the exogenous crisis windows in `_shared.CRISIS_WINDOWS` correspond to actual stock moves on the tickers we're evaluating. If the shaded regions don't line up with visible drawdowns or volatility spikes, the dates need revisiting before we report adaptation-speed numbers against them.

**Crisis windows (exogenous):**

| Regime | Onset | End | Event |
|---|---|---|---|
| 2008 | 2008-09-15 | 2009-03-31 | Lehman Brothers bankruptcy |
| 2020 | 2020-02-20 | 2020-04-30 | S&P 500 pre-crash peak |
| 2022 | 2022-01-03 | 2022-12-31 | S&P 500 pre-bear peak (before rate-hike cycle) |

**Crash basket tickers** (only ones iterated in `_shared.iterate_experiments`):
- **2008:** AIG, GNW, MGM — financial collapse + casino
- **2020:** MCD, MGM, AZO — in-person consumer / hospitality
- **2022:** SNAP, NFLX, META — ad-revenue and streaming under rate-hike pressure

**Test-window geometry:** each test window = crisis window + 6 months of calm on each side (via `_shared.test_window`). The 6-month post-crisis tail is what adaptation-speed measurement uses to detect recovery. Crisis : non-crisis ratios differ across regimes (16% / 35% / 50%) because the underlying crises had different durations — metrics are reported per-regime so this is unbiased.

Stable/resilient baskets are defined in `data.py` but not iterated. To re-include them as a control group, add them to `CRISIS_WINDOWS` in `_shared.py`.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "requirements.txt").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis._shared import CRISIS_WINDOWS, ensure_data, test_window
from src.data.data import Cols

plt.rcParams["figure.dpi"] = 100


In [ ]:
data = ensure_data()
print(f"data: {len(data):,} rows  |  {data.index.min().date()} -> {data.index.max().date()}")

print("\nExperiment grid (crash baskets only):")
for regime, cfg in CRISIS_WINDOWS.items():
    tickers = ", ".join(t.value for t in cfg["tickers"])
    print(f"  {regime}  onset={cfg['onset'].date()}  end={cfg['end'].date()}  tickers={tickers}")


## Figure 1: Cumulative log return

For each `(regime, ticker)` pair, plots cumulative log return over `[onset - 6mo, end + 6mo]` (via `_shared.test_window`). Steep negative slopes inside the shaded red region (= crisis window) confirm the regime dates capture real drawdowns.

Interpretation: `cum_log_return = -0.5` ≈ 40% price drop. The y-axis is unitless but monotone with drawdown severity.


In [ ]:
def _window_slice(data, label_col, onset, end):
    start, stop = test_window(onset, end)
    return data.loc[(data.index >= start) & (data.index <= stop), label_col]


def plot_regime_grid(data, metric_fn, suptitle, ylabel, figsize=(15, 11)):
    n_rows = len(CRISIS_WINDOWS)
    n_cols = max(len(cfg["tickers"]) for cfg in CRISIS_WINDOWS.values())
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize, squeeze=False)
    fig.suptitle(suptitle, fontsize=13, fontweight="bold")

    for row, (regime, cfg) in enumerate(CRISIS_WINDOWS.items()):
        onset, end = cfg["onset"], cfg["end"]
        for col in range(n_cols):
            ax = axes[row, col]
            if col >= len(cfg["tickers"]):
                ax.axis("off")
                continue
            ticker = cfg["tickers"][col]
            label_col = f"{Cols.LogReturns.value}_{ticker.value}"
            returns = _window_slice(data, label_col, onset, end)
            series = metric_fn(returns)

            ax.plot(series.index, series.values, color="steelblue", linewidth=1.1)
            ax.axvspan(onset, end, alpha=0.18, color="red")
            ax.axvline(onset, color="red", linestyle="--", linewidth=0.8, alpha=0.7)
            ax.axvline(end,   color="red", linestyle="--", linewidth=0.8, alpha=0.7)
            ax.set_title(f"{regime}: {ticker.value}", fontsize=10)
            ax.grid(alpha=0.3)
            if col == 0:
                ax.set_ylabel(ylabel)
            ax.tick_params(axis="x", labelrotation=30, labelsize=8)

    plt.tight_layout()
    return fig


fig1 = plot_regime_grid(
    data,
    metric_fn=lambda returns: returns.cumsum(),
    suptitle="Cumulative log return (crisis window shaded)",
    ylabel="Cumulative log return",
)
plt.show()


## Figure 2: Rolling 20-day realized volatility (annualized)

Same grid, plotting `rolling(20).std() * sqrt(252)`. Spikes inside the shaded region confirm the volatility regime shift — which is what the `high_vol_mask` in Metrics.md is designed to capture.

If a ticker's vol barely changes inside the crisis window, that (ticker, regime) pair is a weak test case and you may want to drop it or weigh it less in the headline metrics.


In [ ]:
fig2 = plot_regime_grid(
    data,
    metric_fn=lambda returns: returns.rolling(20).std() * np.sqrt(252),
    suptitle="Rolling 20-day realized volatility, annualized (crisis window shaded)",
    ylabel="Annualized vol",
)
plt.show()


## Numeric summary

Numeric back-up to the plots. For each `(regime, ticker)`:
- **max_drawdown_pct** — largest peak-to-trough percentage drop *within* the crisis window (computed by compounding log returns from the onset date).
- **crisis_vol** / **outside_vol** — annualized realized vol inside vs outside the crisis window, computed over the same test window used for the plots.
- **vol_ratio** — `crisis_vol / outside_vol`. Values >> 1 mean the crisis window is genuinely more volatile for this ticker; values near 1 would be a red flag that the regime dates don't bite for that ticker.


In [ ]:
rows = []
for regime, cfg in CRISIS_WINDOWS.items():
    onset, end = cfg["onset"], cfg["end"]
    for ticker in cfg["tickers"]:
        label_col = f"{Cols.LogReturns.value}_{ticker.value}"
        returns = _window_slice(data, label_col, onset, end)

        in_crisis  = returns[(returns.index >= onset) & (returns.index <= end)]
        out_crisis = returns[(returns.index <  onset) | (returns.index >  end)]

        # peak-to-trough drawdown inside the crisis window
        price = np.exp(in_crisis.cumsum())
        running_peak = price.cummax()
        max_drawdown_pct = 100 * ((price / running_peak) - 1).min()

        crisis_vol  = in_crisis.std()  * np.sqrt(252)
        outside_vol = out_crisis.std() * np.sqrt(252)
        vol_ratio   = crisis_vol / outside_vol if outside_vol > 0 else np.nan

        rows.append({
            "regime": regime,
            "ticker": ticker.value,
            "n_crisis": len(in_crisis),
            "n_outside": len(out_crisis),
            "max_drawdown_pct": round(max_drawdown_pct, 1),
            "crisis_vol": round(crisis_vol, 3),
            "outside_vol": round(outside_vol, 3),
            "vol_ratio": round(vol_ratio, 2),
        })

summary = pd.DataFrame(rows).set_index(["regime", "ticker"])
summary


## Notes for interpretation

- **Drawdown** should be large and negative for crash-basket tickers. If any row shows a drawdown less than ~10%, that (ticker, regime) pair isn't a real stress test and may dilute the headline metrics.
- **vol_ratio > 1.5** is a reasonable threshold for "this ticker genuinely experienced a vol shift." Values near 1.0 mean the calm baseline is already volatile and the crisis wasn't distinguishable for this ticker.
- Later, `results.ipynb` will overlay rolling 20-day coverage per model onto these same subplots — so you can see *coverage collapsing* at the same x-axis location where vol spikes.
